In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
EXPECTED_PYTHON = (PROJECT_ROOT / ".venv" / "Scripts" / "python.exe").resolve()

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if Path(sys.executable).resolve() != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Kernel incorrecto. Selecciona 'football-ml (.venv)'. Ejecutable actual: {sys.executable}"
    )

from football_ml.config import load_ingestion_config

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print(f"Python executable: {sys.executable}")
print(f"Project root:      {PROJECT_ROOT}")


In [ ]:
config = load_ingestion_config()
RAW_DIR = config.raw_dir
CANONICAL_FILES = [config.canonical_csv_path(season) for season in config.seasons]
AVAILABLE_FILES = [path for path in CANONICAL_FILES if path.exists()]
MISSING_FILES = [path.name for path in CANONICAL_FILES if not path.exists()]

print(f"Raw dir:          {RAW_DIR}")
print(f"Available files:  {len(AVAILABLE_FILES)}")
if MISSING_FILES:
    print(f"Missing files:    {MISSING_FILES}")

if not AVAILABLE_FILES:
    raise FileNotFoundError(
        "No hay CSV canonicos en data/bronze/matchhistory/raw. Ejecuta .\\scripts\\ingest-matchhistory.ps1 primero."
    )


In [ ]:
frames = []
for season, csv_path in zip(config.seasons, CANONICAL_FILES):
    if not csv_path.exists():
        continue
    frame = pd.read_csv(csv_path)
    frame["season_source"] = season
    frame["source_file"] = csv_path.name
    frames.append(frame)

df_raw = pd.concat(frames, ignore_index=True)

print(f"Shape: {df_raw.shape}")
display(df_raw.head())


In [ ]:
print("Columnas:")
display(pd.DataFrame({"column": df_raw.columns}))

print("Tipos:")
display(df_raw.dtypes.rename("dtype").to_frame())

print("Nulos por columna:")
display(df_raw.isna().sum().rename("nulls").sort_values(ascending=False).to_frame())
